Este netbook realizara un Análisis Exploratorio de Datos (EDA) enfocado en la caracterización estructural del dataset, la evolucion y la identificación de comportamientos relevantes para guiar procesos posteriores de preprocesamiento y construcción de modelos.

In [15]:
""" Importaciones necesarias """
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 0. Configuracion de rutas y configuracion visual

In [16]:
ruta_archivo = Path.cwd().parent / "data" / "raw" / "dataset_comida.csv"

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

## 1. Ingesta de los datos

In [17]:
def cargar_archivo():
    try:
        df = pd.read_csv(ruta_archivo)
        print(f"Carga exitosa del DataFrame, este se conforma de la siguiente manera:\n\nColumnas: {df.shape[1]}\nFilas: {df.shape[0]}")

        return df

    except FileNotFoundError:
        print(f"    ERROR: No se encontro el archivo en {ruta.absolute()}")
        return None

    except Exception as e:
        print(f"    ERROR INESPERADO: {e}")

df = cargar_archivo()

Carga exitosa del DataFrame, este se conforma de la siguiente manera:

Columnas: 19
Filas: 50000


## 2. Vista inicial del DataFrame

In [18]:
""" Vista general de las primeras 10 filas """
df.head(10)

,order_id,user_id,age,city,order_time,day_type,cuisine,meal_type,restaurant_type,order_value,discount_applied,delivery_fee,time_taken_to_order,rating_given,is_repeat_order,mood,hunger_level,company,rainy_weather
0,1,2698,35,Pune,Evening,Weekend,Chinese,Dinner,Premium,971,Yes,90,13,1,Yes,Celebrating,High,Partner,No
1,2,3237,44,Mumbai,Night,Weekend,South Indian,Dinner,Budget,442,No,26,13,2,No,Lazy,Low,Family,No
2,3,3626,31,Delhi,Morning,Weekend,Biryani,Breakfast,Mid-range,739,Yes,85,10,2,Yes,Happy,Medium,Friends,No
3,4,3176,23,Delhi,Evening,Weekend,Biryani,Snacks,Mid-range,466,No,44,12,2,No,Happy,Medium,Alone,No
4,5,4824,26,Chandigarh,Morning,Weekday,Chinese,Lunch,Premium,927,Yes,58,13,2,No,Happy,Medium,Partner,Yes
5,6,2854,33,Mumbai,Afternoon,Weekend,Biryani,Dinner,Budget,207,No,42,1,1,No,Stressed,High,Partner,No
6,7,3019,33,Bangalore,Evening,Weekday,Chinese,Lunch,Budget,565,No,28,1,4,Yes,Stressed,Low,Alone,No
7,8,1505,24,Mumbai,Evening,Weekday,Fast Food,Lunch,Budget,598,Yes,51,10,2,Yes,Happy,Medium,Partner,Yes
8,9,3157,37,Mumbai,Night,Weekend,North Indian,Breakfast,Mid-range,469,No,29,8,3,Yes,Celebrating,Medium,Friends,No
9,10,1704,21,Pune,Afternoon,Weekday,Biryani,Dinner,Mid-range,549,No,51,6,2,Yes,Lazy,Medium,Alone,Yes


In [19]:
""" Muestra total de las filas y columnas, el tipo de datos (dtypes) de las columnas """
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   order_id             50000 non-null  int64 
 1   user_id              50000 non-null  int64 
 2   age                  50000 non-null  int64 
 3   city                 50000 non-null  object
 4   order_time           50000 non-null  object
 5   day_type             50000 non-null  object
 6   cuisine              50000 non-null  object
 7   meal_type            50000 non-null  object
 8   restaurant_type      50000 non-null  object
 9   order_value          50000 non-null  int64 
 10  discount_applied     50000 non-null  object
 11  delivery_fee         50000 non-null  int64 
 12  time_taken_to_order  50000 non-null  int64 
 13  rating_given         50000 non-null  int64 
 14  is_repeat_order      50000 non-null  object
 15  mood                 50000 non-null  object
 16  hung

Observacion:

    - No se observan valores nulos en el.
    
    - Las columnas discount_applied / is_repeat_order / rainy_weather son de respuesta "si o no", convertirlas en 0 y 1 para poder facilitar el calculo de correlacion.
    
    - Variables de tiempo: order_time aunque el tipo es object, contiene categorias claras. No es una "estampa de tiempo" con horas exactas, sino una variable categorica ordinal.

    - Variables numéricas: order_id, user_id, rating_given, estan correctamente tipadas.

    - Variables Binarias: Columnas como discount_applied, is_repeat_order y rainy_weather están como texto (Yes/No). Para optimizar modelos, deberán convertirse a tipo bool o int (1 / 0).

In [20]:
round(df.describe())

,order_id,user_id,age,order_value,delivery_fee,time_taken_to_order,rating_given
count,50000.0,50000.0,50000.0,50000.0,50000.0,50000.0,50000.0
mean,25000.0,3005.0,31.0,548.0,60.0,7.0,3.0
std,14434.0,1152.0,8.0,260.0,23.0,4.0,1.0
min,1.0,1000.0,18.0,100.0,20.0,1.0,1.0
25%,12501.0,2013.0,24.0,323.0,40.0,4.0,2.0
50%,25000.0,3004.0,31.0,547.0,60.0,7.0,3.0
75%,37500.0,4002.0,38.0,772.0,80.0,11.0,4.0
max,50000.0,4999.0,44.0,999.0,99.0,14.0,5.0


## 3. Deteccion de valores nulos y duplicados

In [26]:
valores_nulos = df.isnull().sum()
nulos_pct = (df.isnull().sum() / len(df)) * 100

# DataFrame de reporte 
df_nulos = pd.DataFrame({
    "Nulos": valores_nulos,
    "Porcentaje nulos": nulos_pct
})

reporte_errores = df_nulos[df_nulos["Nulos"] >0].sort_values(by= "Porcentaje nulos", ascending=False)

print(f"Total de registros: {len(df)}")

if not reporte_errores.empty:
    print("Columnas requieren atencion de inmediato.")
    print(reporte_errores)
else:
    print("\nNo se encontraron valores nulos en el dataset.")

Total de registros: 50000

No se encontraron valores nulos en el dataset.


In [24]:
valores_duplicados = df.duplicated().sum()

print(f"Cantidad de valores duplicados: {valores_duplicados}")

Cantidad de valores duplicados: 0


In [28]:
""" Pedidos realizados segun el animo de cada persona """

print(df['mood'].value_counts())

mood
Stressed       12672
Celebrating    12580
Happy          12465
Lazy           12283
Name: count, dtype: int64


Equilibrio de Clases: La muestra es altamente equitativa, con una diferencia menor al 3% entre la categoría más frecuente (Stressed) y la menos frecuente (Lazy). Esto elimina la necesidad de técnicas de sobremuestreo (oversampling) en futuros modelos.

Independencia del Hábito: El volumen de pedidos no parece estar condicionado por el estado anímico, lo que posiciona a la aplicación como un servicio de necesidad regular y no estacional.

Hipótesis a validar: Se requiere investigar si existe una correlación entre el mood y el cuisine (estilo de comida), para determinar si el ánimo influye en la elección del producto, aunque no influya en la decisión de pedir.

## 4. Análisis Bivariado y cruce de variables

### 4.1 Relacion entre el estado de ánimo y gastos

In [33]:
estado_animo_gastos = df.groupby('mood')['order_value'].agg(Promedio = 'mean', Variacion = "std")
display(estado_animo_gastos.round(2))

,Promedio,Variacion
mood,,
Celebrating,547.39,259.40
Happy,545.51,259.56
Lazy,550.48,260.39
Stressed,547.59,259.42


- Homogeneidad del gasto: La diferencia entre el que mas gasta y el que menos gaste es apenas de $5. 
El estado de animo no influye en el presupuesto que el usuario asigna a su comida. La gente gasta lo mismo sin importar cómo se sienta.

- Desviación estándar: En todos los grupos es casi calcada, el comportamiento de gasto es muy estable en todos los perfiles emocionales. No hay grupo que sea mas "imporedecible" que otro.



### 4.2 Preferencia de comidas por el humor

In [34]:
preferencia_comida_humor =  pd.crosstab(df['mood'], df['cuisine'], normalize= 'index') * 100
display(preferencia_comida_humor.round(2))

cuisine,Biryani,Chinese,Desserts,Fast Food,North Indian,South Indian
mood,,,,,,
Celebrating,16.54,16.87,16.69,16.34,16.33,17.23
Happy,16.61,16.29,17.12,16.96,16.59,16.44
Lazy,16.76,16.36,17.16,17.15,16.30,16.26
Stressed,16.49,17.27,16.65,16.51,16.60,16.47


Conclusión del cruce Mood vs Cuisine:

Se observa una neutralidad absoluta en las preferencias gastronómicas segun el esado anímico. No existe una "comida preferida" para los momentos de esters ni para las celebraciones. La probabilidad de elegir cualquier tipo de cocina se mantiene constante (~16.6%) en todos los perfiles psicologicos.

Dado que el 'mood' no influye ni en el gasto (order_value) ni en el tipo de comida (cuisine), esta variable podria no ser un predictor fuerte para un modelo de recomendación, aunque sigue siendo útil para caracterizar al usuario.